# RL2307_calibration_standardize

Standardize the RL2307 manufacturer calibration files and return them for review.

**Author:** <Your Name>
**Version:** 1.0

In [ ]:
# Uncomment the lines below to install required packages.
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_Utils.git
# %pip install git+https://github.com/BLayman-NOAA/AA-SI_calibration.git

In [ ]:
from aa_si_utils.utils import initial_setup_and_validation
from aa_si_calibration.calibration import read_raw_file_config
from aa_si_calibration.calibration import record_raw_file_configs
from aa_si_calibration.calibration import standardize_calibration_files


In [ ]:
raw_input_folder = 'gs://ggn-nmfs-aa-prod-1-data/HDD/Reuben_Lasker/RL2307/EK80/data/raw'  # path
cal_input_folder = 'gs://ggn-nmfs-aa-prod-1-data/HDD/Reuben_Lasker/RL2307/EK80/Calibration/RESULTS'  # path
calibration_outputs = 'calibration'  # str
cruise_id = 'RL2307'  # str
record_author = '<Your Name>'  # str
file_time_start = '2023-07-17T16:30:46'  # str
file_time_end = '2023-07-18T19:50:19'  # str
override_channels = None  # TODO: set this value  # dict

## Step: `initial_setup`
**Op:** `initial_setup`

Validate the raw files in the input folder, create the calibration output directory if needed, and return the raw file paths and that directory.

**Parameters:**
- `raw_input_folder` path - Folder of .raw files, local or a remote URL such as gs://bucket/survey/raw. Remote folders are listed without downloading any data.

- `calibration_outputs` str - Subdirectory under the run's outputs folder where calibration artifacts are written, with JSON logs in its logs/ subfolder.
- `raw_file_names` list - Explicit file name list; empty means include all files in folder
- `clear_previous_json_logs` bool - Delete the calibration_flags.json left by a previous run before this one starts, so the flags reflect this run alone. Set false to let the flags accumulate across runs.

- `file_time_start` str - Inclusive ISO datetime, for example "2016-07-25T20:58"; keeps only raw files whose filename time (D{YYYYMMDD}-T{HHMMSS}) is at or after it. Unset keeps all files.
- `file_time_end` str - Optional inclusive ISO datetime upper bound for filename-encoded file times. See file_time_start.

In [ ]:
# --- Parameters ---
_initial_setup__raw_input_folder = 'gs://ggn-nmfs-aa-prod-1-data/HDD/Reuben_Lasker/RL2307/EK80/data/raw'
_initial_setup__calibration_outputs = 'calibration'
_initial_setup__clear_previous_json_logs = True
_initial_setup__file_time_start = '2023-07-17T16:30:46'
_initial_setup__file_time_end = '2023-07-18T19:50:19'

_result = initial_setup_and_validation(
    raw_input_folder=_initial_setup__raw_input_folder,
    calibration_outputs_string=_initial_setup__calibration_outputs,
    clear_previous_json_logs=_initial_setup__clear_previous_json_logs,
    file_time_start=_initial_setup__file_time_start,
    file_time_end=_initial_setup__file_time_end,
)
raw_file_paths = _result['raw_file_paths']
calibration_output_dir = _result['calibration_output_dir']

## Step: `scan_raw_config`
**Op:** `read_raw_file_config`

Read one raw file's channel configuration from its datagram headers. Map it over the raw file paths and collect the results with record_raw_file_configs.

**Parameters:**
- `raw_file_path` path - One .raw file, as a local path or a remote URL. Normally wired to ${_item}.

- `verify_start_time` bool - If true, also read EK80 files in full to verify metadata_start_time, which is slower. The default single pass scan is enough for channel configuration.

- `verbose` bool - Print per-file progress. Set false on a long fan-out to keep the log readable.

- `max_scan_bytes` int - Most leading bytes of a remote EK60 or EK80 file to read instead of the whole file, in bytes. Unset reads whole files. Around 1 MiB suits EK60; FM EK80 needs tens of MiB. Too low only costs a whole-file read.


*Runs once per `map_over` instance (fan-out loop).*

In [ ]:
file_config = []
for _item in raw_file_paths:
    _iter_file_config = read_raw_file_config(
        raw_file_path=_item,
        verbose=False,
        max_scan_bytes=134217728,
    )
    file_config.append(_iter_file_config)

## Step: `record_raw_configs`
**Op:** `record_raw_file_configs`

Collect per-file channel configurations into raw_file_configs/raw_file_configs.yaml under the calibration output directory. Use collect on the scan step's file_config output.

**Parameters:**
- `output_base` path - Root directory the calibration artifacts are written under. Normally wired to initial_setup's calibration_output_dir.

- `verbose` bool - Print a summary of what was recorded.

In [ ]:
# --- Parameters ---
_record_raw_configs__output_base = calibration_output_dir

_result = record_raw_file_configs(file_configs=file_config, output_base=_record_raw_configs__output_base)
raw_file_configs = _result['raw_file_configs']
frequencies = _result['frequencies']
raw_configs_path = _result['raw_configs_path']

## Step: `standardize_cal`
**Op:** `standardize_calibration_files`

Convert manufacturer calibration files (EK60 .cal or EK80 .xml) to standardized single-channel .yaml files under single_channel_calibration_files/. Skipped when the manufacturer files are unchanged.

**Parameters:**
- `cal_input_folder` path - Folder of manufacturer calibration files (.cal for EK60, .xml for EK80), local or a remote URL.

- `output_base` path - Root directory the calibration artifacts are written under. Normally wired to initial_setup's calibration_output_dir.

- `cruise_id` str - Cruise identifier recorded in every generated single-channel file (e.g. "HB2407"), so a standardized file stays traceable to the survey it came from.

- `record_author` str - Name recorded as the author of every generated single-channel file. Not used in any calculation.

- `short_filenames` bool - Use compact names for the generated single-channel files. Set false for longer, more descriptive names.

- `overwrite` bool - Re-parse and rewrite even when the sidecar says the existing files are current, forcing a clean rebuild of the single-channel folder.

- `verbose` bool - Print the parsed parameters and the files written.
- `override_channels` dict - Write these channels as the single-channel files instead of parsing the manufacturer files. Same shape as the single_channel_data output: omit an entry to discard that channel, and edit values in place to correct them.


In [ ]:
# --- Parameters ---
_standardize_cal__cal_input_folder = 'gs://ggn-nmfs-aa-prod-1-data/HDD/Reuben_Lasker/RL2307/EK80/Calibration/RESULTS'
_standardize_cal__output_base = calibration_output_dir
_standardize_cal__cruise_id = 'RL2307'
_standardize_cal__record_author = '<Your Name>'
_standardize_cal__short_filenames = True
_standardize_cal__override_channels = override_channels

_result = standardize_calibration_files(
    frequencies=frequencies,
    cal_input_folder=_standardize_cal__cal_input_folder,
    output_base=_standardize_cal__output_base,
    cruise_id=_standardize_cal__cruise_id,
    record_author=_standardize_cal__record_author,
    short_filenames=_standardize_cal__short_filenames,
    override_channels=_standardize_cal__override_channels,
)
single_channel_dir = _result['single_channel_dir']
channel_count = _result['channel_count']
skipped = _result['skipped']
single_channel_data = _result['single_channel_data']